In [17]:
from langchain.chat_models import init_chat_model
import requests
import os 
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent

In [18]:
llm=init_chat_model(
    model="gemma4:31b-cloud",
    model_provider="ollama"
)

In [19]:
llm.invoke("what is pen")

AIMessage(content='Depending on the context, a **pen** can refer to several different things. Here are the most common meanings:\n\n### 1. The Writing Instrument\nThe most common definition is a handheld tool used to apply ink to a surface, usually paper, for writing or drawing. There are several types:\n*   **Ballpoint Pen:** Uses a small rotating ball to disperse oil-based ink. It is the most common and long-lasting.\n*   **Rollerball Pen:** Similar to a ballpoint but uses water-based liquid ink, which flows more smoothly.\n*   **Fountain Pen:** Uses a metal nib and a reservoir of liquid ink. These are often used for calligraphy or formal writing.\n*   **Gel Pen:** Uses ink where pigment is suspended in a water-based gel, offering a vivid color and smooth feel.\n*   **Stylus/Digital Pen:** A plastic device used to write on touchscreens or tablets.\n\n### 2. An Enclosure for Animals\nA pen is also a small fenced-in area used to keep animals contained. \n*   **Example:** A "pig pen" or

In [20]:
from langchain.tools import tool


In [21]:
@tool
def get_current_weather(city: str) -> str:
    """Fetches real-time weather metrics for a specified city."""
    url = "https://openweathermap.org"
    
    api_key = os.getenv("OPENWEATHER_API_KEY")
    if not api_key:
        return "Error: OPENWEATHER_API_KEY environment variable is not set."

    params = {"q": city, "appid": api_key, "units": "metric"}
    
    try:
        res = requests.get(url, params=params)
        
        # 1. This catches 401, 404, or 500 errors immediately instead of crashing
        res.raise_for_status() 
        
        # 2. Safely parse JSON once we know the request succeeded
        data = res.json()
        return f"Weather in {city}: {data['main']['temp']}°C, {data['weather']['description']}."
        
    except requests.exceptions.HTTPError as http_err:
        status = res.status_code
        if status == 401:
            return "Error 401: Invalid API key. Note: New keys take up to 2 hours to activate."
        elif status == 404:
            return f"Error 404: City '{city}' not found. Check spelling."
        else:
            return f"HTTP error occurred: {status} - {res.text[:100]}"
            
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"


In [22]:
agent=create_agent(model=llm,tools=[get_current_weather],system_prompt="Act as an Helpful Assistant")

In [23]:
response = agent.invoke({"messages": [{"role": "user", "content": "What is the weather in Pune?"}]})


In [25]:
print(response['messages'][-1].content)

I'm sorry, I'm having trouble retrieving the real-time weather data for Pune right now. Please try again in a few moments!
